In [1]:
"""
ATP Tennis Match Data Import
-----------------------------
Downloads and processes ATP tour-level match results from Jeff Sackmann's
tennis_atp GitHub repository (github.com/JeffSackmann/tennis_atp).

Covers seasons 2000-2022, matching the data used in the
Bradley-Terry Stochastic Block Model paper (Santi et al., 2025).

License: Data is CC BY-NC-SA 4.0. Attribution required for academic use.
Citation: Jeff Sackmann, tennis_atp, github.com/JeffSackmann/tennis_atp
"""

import pandas as pd
import numpy as np
from io import StringIO

try:
    import requests
except ImportError:
    raise ImportError("Please install requests: pip install requests")


# ──────────────────────────────────────────────
# 1.  Download raw match data
# ──────────────────────────────────────────────

BASE_URL = (
    "https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/"
    "atp_matches_{year}.csv"
)

def download_season(year: int) -> pd.DataFrame:
    """Download one season's tour-level main draw results."""
    url = BASE_URL.format(year=year)
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pd.read_csv(StringIO(response.text), low_memory=False)
    df["season"] = year
    return df


def download_all_seasons(start: int = 2000, end: int = 2022) -> pd.DataFrame:
    """Download and concatenate multiple seasons."""
    frames = []
    for year in range(start, end + 1):
        print(f"  Downloading {year}...", end=" ")
        try:
            df = download_season(year)
            frames.append(df)
            print(f"{len(df)} matches")
        except requests.HTTPError as e:
            print(f"FAILED ({e})")
    return pd.concat(frames, ignore_index=True)


# ──────────────────────────────────────────────
# 2.  Clean and filter
# ──────────────────────────────────────────────

def clean_matches(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only tour-level singles main draw matches.
    Removes Davis Cup (team competition), retirements/walkovers,
    and matches with missing player names.
    """
    # Drop Davis Cup ties (draw_size == 0 or tourney_level == 'D')
    if "tourney_level" in df.columns:
        df = df[df["tourney_level"] != "D"].copy()

    # Drop walkovers and retirements — score contains 'W/O' or 'RET'
    if "score" in df.columns:
        df = df[
            df["score"].notna()
            & ~df["score"].str.contains("W/O|RET|DEF|In Progress", na=True)
        ].copy()

    # Drop rows with missing player names
    df = df[df["winner_name"].notna() & df["loser_name"].notna()].copy()

    return df.reset_index(drop=True)


# ──────────────────────────────────────────────
# 3.  Select key columns
# ──────────────────────────────────────────────

KEY_COLUMNS = [
    "season",
    "tourney_id",
    "tourney_name",
    "tourney_date",
    "tourney_level",   # G=Grand Slam, M=Masters, A=ATP500/250, etc.
    "surface",
    "round",
    "winner_id",
    "winner_name",
    "winner_rank",
    "loser_id",
    "loser_name",
    "loser_rank",
    "score",
]

def select_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only the columns that exist in the dataset."""
    cols = [c for c in KEY_COLUMNS if c in df.columns]
    return df[cols].copy()


# ──────────────────────────────────────────────
# 4.  Build pairwise comparison table
#     (the direct input format for BT models)
# ──────────────────────────────────────────────

def build_pairwise(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reshape matches into a tidy pairwise comparison table.
    Each row represents one match outcome:
        winner beats loser  →  (season, winner, loser)
    """
    pairs = df[[
        "season", "tourney_name", "surface", "round",
        "winner_name", "winner_rank",
        "loser_name",  "loser_rank",
    ]].rename(columns={
        "winner_name": "winner",
        "winner_rank": "winner_rank",
        "loser_name":  "loser",
        "loser_rank":  "loser_rank",
    }).copy()
    return pairs.reset_index(drop=True)


# ──────────────────────────────────────────────
# 5.  Filter to top-N players per season
#     (BT-SBM paper uses top 105 per season)
# ──────────────────────────────────────────────

def top_players_per_season(
    df: pd.DataFrame, n: int = 105
) -> pd.DataFrame:
    """
    Restrict to matches where both players are in the
    top-n ranked players at the start of that season.

    Uses the minimum rank observed during the season as a proxy
    when season-start rankings are unavailable.
    """
    # Collect all (season, player, rank) observations
    winners = df[["season", "winner_name", "winner_rank"]].rename(
        columns={"winner_name": "player", "winner_rank": "rank"}
    )
    losers = df[["season", "loser_name", "loser_rank"]].rename(
        columns={"loser_name": "player", "loser_rank": "rank"}
    )
    all_ranks = pd.concat([winners, losers]).dropna(subset=["rank"])
    all_ranks["rank"] = pd.to_numeric(all_ranks["rank"], errors="coerce")

    # Best (lowest) rank per player per season
    best_rank = (
        all_ranks.groupby(["season", "player"])["rank"]
        .min()
        .reset_index()
    )

    # Top-n players per season
    top_n = (
        best_rank[best_rank["rank"] <= n]
        .groupby("season")["player"]
        .apply(set)
        .to_dict()
    )

    # Keep matches where both players were in top-n
    mask = df.apply(
        lambda row: (
            row["winner_name"] in top_n.get(row["season"], set())
            and row["loser_name"] in top_n.get(row["season"], set())
        ),
        axis=1,
    )
    return df[mask].reset_index(drop=True)



# ──────────────────────────────────────────────
# 6.  Summary statistics
# ──────────────────────────────────────────────

def summarise(df: pd.DataFrame) -> None:
    print("\n── Dataset Summary ──────────────────────────────")
    print(f"  Seasons:  {df['season'].min()} – {df['season'].max()}")
    print(f"  Matches:  {len(df):,}")
    print(f"  Players:  {pd.concat([df['winner_name'], df['loser_name']]).nunique():,}")
    print(f"  Seasons with data: {df['season'].nunique()}")
    if "surface" in df.columns:
        print(f"\n  Matches by surface:")
        print(df["surface"].value_counts().to_string(index=True))
    print("─────────────────────────────────────────────────\n")


# ──────────────────────────────────────────────
# 7.  Main pipeline
# ──────────────────────────────────────────────

def load_atp_data(
    start: int = 2000,
    end:   int = 2022,
    top_n: int = 105,        # set None to keep all players
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Full pipeline. Returns:
        matches  – cleaned match-level DataFrame
        pairs    – tidy pairwise comparison DataFrame
    """
    print(f"Downloading ATP data {start}–{end} from JeffSackmann/tennis_atp...")
    raw = download_all_seasons(start, end)

    print("\nCleaning...")
    matches = clean_matches(raw)
    matches = select_columns(matches)

    if top_n is not None:
        print(f"Filtering to top {top_n} players per season...")
        matches = top_players_per_season(matches, n=top_n)

    pairs = build_pairwise(matches)
    summarise(matches)

    return matches, pairs


# ──────────────────────────────────────────────
# 8.  Example usage
# ──────────────────────────────────────────────

if __name__ == "__main__":

    # Load the full 2000-2022 dataset (replicating the BT-SBM paper)
    matches, pairs = load_atp_data(start=2000, end=2022, top_n=105)

    print("matches.head():")
    print(matches.head())
    print("\npairs.head():")
    print(pairs.head())

    # ── Access a single season ────────────────────────────────────────
    season_2017 = matches[matches["season"] == 2017]
    print(f"\n2017 season: {len(season_2017)} matches")

    # ── Win/loss counts per player in 2017 ───────────────────────────
    wins_2017   = season_2017["winner_name"].value_counts().rename("wins")
    losses_2017 = season_2017["loser_name"].value_counts().rename("losses")
    record_2017 = pd.concat([wins_2017, losses_2017], axis=1).fillna(0).astype(int)
    record_2017["total"] = record_2017["wins"] + record_2017["losses"]
    record_2017["win_pct"] = (record_2017["wins"] / record_2017["total"]).round(3)
    print("\nTop 10 players by win % in 2017 (min 10 matches):")
    print(
        record_2017[record_2017["total"] >= 10]
        .sort_values("win_pct", ascending=False)
        .head(10)
    )

    # ── Save to CSV ───────────────────────────────────────────────────
    matches.to_csv("atp_matches_2000_2022.csv", index=False)
    pairs.to_csv("atp_pairs_2000_2022.csv", index=False)
    print("\nSaved: atp_matches_2000_2022.csv, atp_pairs_2000_2022.csv")


Cleaning...
Filtering to top 105 players per season...

── Dataset Summary ──────────────────────────────
  Seasons:  2000 – 2022
  Matches:  47,587
  Players:  544
  Seasons with data: 23

  Matches by surface:
surface
Hard      26368
Clay      15148
Grass      4920
Carpet     1151
─────────────────────────────────────────────────

matches.head():
   season tourney_id tourney_name  tourney_date tourney_level surface round  \
0    2000   2000-301     Auckland      20000110             A    Hard   R32   
1    2000   2000-301     Auckland      20000110             A    Hard   R32   
2    2000   2000-301     Auckland      20000110             A    Hard   R32   
3    2000   2000-301     Auckland      20000110             A    Hard   R32   
4    2000   2000-301     Auckland      20000110             A    Hard   R32   

   winner_id          winner_name  winner_rank  loser_id           loser_name  \
0     103163           Tommy Haas         11.0    101543         Jeff Tarango   
1     10260

In [9]:
"""
ATP Pairwise Data → Complete Strict Rankings
----------------------------------------------
Converts ATP match outcomes (pairwise comparisons) into complete strict
rankings of players per season, using three methods:

    1. Borda count          – simple win count, fast baseline
    2. Win percentage       – accounts for number of matches played
    3. Bradley-Terry MLE    – principled probabilistic ranking
       (requires the 'choix' package: pip install choix)

Each method produces one ranking per season: a strict ordering of the
top-N players from best (rank 1) to worst (rank N).

Input:  pairs DataFrame from atp_import.py  (or the saved CSV)
Output: rankings DataFrame  (season × player → rank)
        rankings matrix     (N_seasons × N_players) for Mallows models
"""

import warnings
import numpy as np
import pandas as pd


# ──────────────────────────────────────────────
# 1.  Borda Count
# ──────────────────────────────────────────────

def borda_ranking(pairs_season: pd.DataFrame) -> pd.Series:
    """
    For one season: rank players by total number of wins (Borda score).
    Players with more wins get a lower rank number (rank 1 = best).
    Ties in win count are broken by losses (fewer losses = better),
    then alphabetically for full determinism.
    """
    wins   = pairs_season["winner"].value_counts().rename("wins")
    losses = pairs_season["loser"].value_counts().rename("losses")

    scores = pd.concat([wins, losses], axis=1).fillna(0).astype(int)
    scores.index.name = "player"

    # Sort: most wins first, then fewest losses, then name
    scores = scores.sort_values(
        ["wins", "losses", scores.index.name],
        ascending=[False, True, True]
    )

    # Assign strict ranks 1..N  (no ties — this is a strict ranking)
    scores["rank"] = np.arange(1, len(scores) + 1)
    return scores["rank"]


# ──────────────────────────────────────────────
# 2.  Win Percentage
# ──────────────────────────────────────────────

def winpct_ranking(pairs_season: pd.DataFrame,
                   min_matches: int = 5) -> pd.Series:
    """
    Rank by win percentage = wins / (wins + losses).
    Players with fewer than min_matches total are dropped.
    Ties broken by total matches played (more = better), then name.
    """
    wins   = pairs_season["winner"].value_counts().rename("wins")
    losses = pairs_season["loser"].value_counts().rename("losses")

    scores = pd.concat([wins, losses], axis=1).fillna(0).astype(int)
    scores.index.name = "player"
    scores["total"]   = scores["wins"] + scores["losses"]
    scores["win_pct"] = scores["wins"] / scores["total"]

    scores = scores[scores["total"] >= min_matches].copy()
    scores = scores.sort_values(
        ["win_pct", "total"],
        ascending=[False, False]
    )
    scores["rank"] = np.arange(1, len(scores) + 1)
    return scores["rank"]


# ──────────────────────────────────────────────
# 3.  Bradley-Terry MLE  (via choix)
# ──────────────────────────────────────────────

def bt_ranking(pairs_season: pd.DataFrame) -> pd.Series | None:
    """
    Fit a Bradley-Terry model by MLE and rank players by their
    estimated log-strength parameters (higher = better = lower rank).

    Requires: pip install choix
    Returns None if choix is not installed or fitting fails.
    """
    try:
        import choix
    except ImportError:
        warnings.warn(
            "choix not installed. Skipping Bradley-Terry ranking.\n"
            "Install with: pip install choix",
            stacklevel=2,
        )
        return None

    players = sorted(
        set(pairs_season["winner"]) | set(pairs_season["loser"])
    )
    player_idx = {p: i for i, p in enumerate(players)}
    n = len(players)

    # Build list of (winner_idx, loser_idx) tuples
    comparisons = [
        (player_idx[row.winner], player_idx[row.loser])
        for row in pairs_season.itertuples()
    ]

    try:
        # MM algorithm for BTL MLE
        params = choix.mm_pairwise(n, comparisons, alpha=1e-4)
    except Exception as e:
        warnings.warn(f"BT fitting failed: {e}", stacklevel=2)
        return None

    strength = pd.Series(params, index=players, name="bt_strength")
    # Higher strength → lower rank number
    ranked = strength.sort_values(ascending=False)
    rank_series = pd.Series(
        np.arange(1, len(ranked) + 1),
        index=ranked.index,
        name="rank"
    )
    return rank_series


# ──────────────────────────────────────────────
# 4.  Apply across all seasons
# ──────────────────────────────────────────────

def rank_all_seasons(
    pairs: pd.DataFrame,
    method: str = "borda",
    top_n: int | None = None,
    **kwargs,
) -> pd.DataFrame:
    """
    Apply a ranking method to every season independently.

    Parameters
    ----------
    pairs   : pairwise DataFrame with columns [season, winner, loser]
    method  : 'borda' | 'winpct' | 'bt'
    top_n   : if set, keep only the top-N ranked players per season
    **kwargs: passed to the underlying ranking function

    Returns
    -------
    DataFrame with columns [season, player, rank]
    """
    method_fn = {
        "borda":  borda_ranking,
        "winpct": winpct_ranking,
        "bt":     bt_ranking,
    }
    if method not in method_fn:
        raise ValueError(f"method must be one of {list(method_fn)}")

    fn = method_fn[method]
    records = []

    for season, grp in pairs.groupby("season"):
        rank_series = fn(grp, **kwargs)
        if rank_series is None:
            continue  # BT failed for this season
        for player, rank in rank_series.items():
            records.append({"season": season, "player": player, "rank": rank})

    df = pd.DataFrame(records)

    if df.empty:
        return df

    if top_n is not None:
        df = df[df["rank"] <= top_n].copy()
        # Re-number ranks 1..top_n within each season
        df["rank"] = df.groupby("season")["rank"].rank(method="first").astype(int)

    return df.sort_values(["season", "rank"]).reset_index(drop=True)


# ──────────────────────────────────────────────
# 5.  Pivot to assessor × item matrix
#     (the format expected by Mallows models)
# ──────────────────────────────────────────────

def to_rank_matrix(
    rankings: pd.DataFrame,
    assessor_col: str = "season",
    item_col: str = "player",
    rank_col: str = "rank",
) -> tuple[np.ndarray, list, list]:
    """
    Pivot the long-format rankings table into a matrix
    of shape (n_assessors × n_items) where entry [i, j]
    is the rank assigned to item j by assessor i.

    In the ATP context:
        assessors = seasons  (each season 'votes' on the player ordering)
        items     = players

    Only players that appear in ALL seasons are included,
    ensuring a complete ranking matrix with no missing values.
    This matches the strict complete ranking requirement of
    Mallows mixture models.

    Returns
    -------
    matrix      : np.ndarray  shape (n_seasons, n_players)
    assessors   : list of season labels (row index)
    items       : list of player names  (column index)
    """
    pivot = rankings.pivot(
        index=assessor_col,
        columns=item_col,
        values=rank_col,
    )

    # Keep only players present in every season
    complete_mask = pivot.notna().all(axis=0)
    n_dropped = (~complete_mask).sum()
    if n_dropped > 0:
        print(
            f"  Dropping {n_dropped} players not present in all seasons "
            f"(keeping {complete_mask.sum()} players)."
        )
    pivot = pivot.loc[:, complete_mask]

    matrix    = pivot.values.astype(int)
    assessors = list(pivot.index)
    items     = list(pivot.columns)

    return matrix, assessors, items


# ──────────────────────────────────────────────
# 6.  Alternatively: treat each season separately
#     (if you don't need a shared item set)
# ──────────────────────────────────────────────

def to_season_matrices(
    rankings: pd.DataFrame,
) -> dict[int, tuple[np.ndarray, list]]:
    """
    Return a dict mapping season → (rank_vector_array, player_list).
    Each season produces one ranking (a single row / one assessor).
    Useful if you want to concatenate rankings across multiple seasons
    with potentially different player pools.
    """
    result = {}
    for season, grp in rankings.groupby("season"):
        grp_sorted = grp.sort_values("rank")
        players = list(grp_sorted["player"])
        ranks   = list(grp_sorted["rank"])
        result[season] = (np.array(ranks), players)
    return result


# ──────────────────────────────────────────────
# 7.  Diagnostics: how consistent are rankings
#     across seasons? (Kendall's W)
# ──────────────────────────────────────────────

def kendalls_w(matrix: np.ndarray) -> float:
    """
    Kendall's coefficient of concordance W.
    W = 1  → perfect agreement across all assessors (seasons)
    W = 0  → no agreement
    """
    n_assessors, n_items = matrix.shape
    # Sum of ranks per item across assessors
    col_sums = matrix.sum(axis=0)
    mean_col_sum = col_sums.mean()
    ss = np.sum((col_sums - mean_col_sum) ** 2)
    w = (12 * ss) / (n_assessors ** 2 * (n_items ** 3 - n_items))
    return w


# ──────────────────────────────────────────────
# 8.  Main — example usage
# ──────────────────────────────────────────────

if __name__ == "__main__":

    # ── Load pairs (either from atp_import.py or saved CSV) ──────────
    try:
        pairs = pd.read_csv("atp_pairs_2000_2022.csv")
        print(f"Loaded {len(pairs):,} pairwise comparisons from CSV.")
    except FileNotFoundError:
        print("atp_pairs_2000_2022.csv not found.")
        print("Run atp_import.py first, or run this combined:")
        print()
        print("  from atp_import import load_atp_data")
        print("  matches, pairs = load_atp_data(start=2000, end=2022, top_n=105)")
        raise

    # ── 1. Borda count rankings ───────────────────────────────────────
    print("\n── Borda Count Rankings ─────────────────────────")
    borda = rank_all_seasons(pairs, method="borda", top_n=30)
    print(borda[borda["season"] == 2017].head(10).to_string(index=False))

    # ── 2. Win percentage rankings ────────────────────────────────────
    print("\n── Win Percentage Rankings ──────────────────────")
    winpct = rank_all_seasons(pairs, method="winpct", min_matches=5, top_n=30)
    print(winpct[winpct["season"] == 2017].head(10).to_string(index=False))

    # ── 3. Bradley-Terry rankings (if choix installed) ────────────────
    print("\n── Bradley-Terry Rankings ───────────────────────")
    bt = rank_all_seasons(pairs, method="bt", top_n=30)
    if bt is not None and len(bt) > 0:
        print(bt[bt["season"] == 2017].head(10).to_string(index=False))
    else:
        print("  Bradley-Terry ranking not available (choix not installed or fitting failed).")

    # ── Convert Borda rankings to rank matrix ─────────────────────────
    print("\n── Rank Matrix (Borda, top 30 players in all seasons) ──")
    matrix, seasons, players = to_rank_matrix(borda)
    print(f"  Matrix shape: {matrix.shape}  (seasons × players)")
    print(f"  Seasons: {seasons[0]} – {seasons[-1]}")
    print(f"  Players (first 5): {players[:5]}")
    print(f"  Kendall's W: {kendalls_w(matrix):.3f}")
    print()
    print("  First 5 rows / 5 cols of matrix:")
    preview = pd.DataFrame(matrix[:5, :5], index=seasons[:5], columns=players[:5])
    print(preview.to_string())

    # ── Save outputs ──────────────────────────────────────────────────
    borda.to_csv("atp_rankings_borda.csv", index=False)
    print("\nSaved: atp_rankings_borda.csv")

    rank_df = pd.DataFrame(matrix, index=seasons, columns=players)
    rank_df.to_csv("atp_rank_matrix_borda.csv")
    print("Saved: atp_rank_matrix_borda.csv  (seasons × players, values = ranks)")
    print(
        "\nNote: atp_rank_matrix_borda.csv is ready to pass directly to a "
        "Mallows mixture model.\nEach row = one assessor (season), "
        "each column = one item (player)."
    )


Loaded 47,587 pairwise comparisons from CSV.

── Borda Count Rankings ─────────────────────────
 season                player  rank
   2017          Rafael Nadal     1
   2017      Alexander Zverev     2
   2017         Roger Federer     3
   2017          David Goffin     4
   2017         Dominic Thiem     5
   2017       Grigor Dimitrov     6
   2017 Roberto Bautista Agut     7
   2017           Marin Cilic     8
   2017             Jack Sock     9
   2017 Juan Martin del Potro    10

── Win Percentage Rankings ──────────────────────
 season           player  rank
   2017    Roger Federer     1
   2017     Rafael Nadal     2
   2017 Filip Krajinovic     3
   2017   Novak Djokovic     4
   2017     Milos Raonic     5
   2017  Grigor Dimitrov     6
   2017 Alexander Zverev     7
   2017      Laslo Djere     8
   2017      Andy Murray     9
   2017    Kei Nishikori    10

── Bradley-Terry Rankings ───────────────────────


C:\Users\jonanord\AppData\Local\Temp\ipykernel_12128\1599167548.py:169: UserWarning: BT fitting failed: Did not converge after 10000 iterations
  rank_series = fn(grp, **kwargs)


 season                player  rank
   2017         Roger Federer     1
   2017          Rafael Nadal     2
   2017        Novak Djokovic     3
   2017      Filip Krajinovic     4
   2017 Juan Martin del Potro     5
   2017          Nick Kyrgios     6
   2017       Grigor Dimitrov     7
   2017      Alexander Zverev     8
   2017          Milos Raonic     9
   2017          David Goffin    10

── Rank Matrix (Borda, top 30 players in all seasons) ──
  Dropping 177 players not present in all seasons (keeping 0 players).
  Matrix shape: (23, 0)  (seasons × players)
  Seasons: 2000 – 2022
  Players (first 5): []
  Kendall's W: nan

  First 5 rows / 5 cols of matrix:
Empty DataFrame
Columns: []
Index: [2000, 2001, 2002, 2003, 2004]

Saved: atp_rankings_borda.csv
Saved: atp_rank_matrix_borda.csv  (seasons × players, values = ranks)

Note: atp_rank_matrix_borda.csv is ready to pass directly to a Mallows mixture model.
Each row = one assessor (season), each column = one item (player).


C:\Users\jonanord\AppData\Local\Temp\ipykernel_12128\1599167548.py:279: RuntimeWarning: Mean of empty slice
  mean_col_sum = col_sums.mean()
c:\Users\jonanord\OneDrive - Universitetet i Oslo\Documents\Code\TiedBayesMallows\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\jonanord\AppData\Local\Temp\ipykernel_12128\1599167548.py:281: RuntimeWarning: invalid value encountered in scalar divide
  w = (12 * ss) / (n_assessors ** 2 * (n_items ** 3 - n_items))


In [10]:
bt_matrix, bt_seasons, bt_players = to_rank_matrix(bt)
print(f"BT matrix shape: {bt_matrix.shape}  (seasons × players)")


  Dropping 164 players not present in all seasons (keeping 0 players).
BT matrix shape: (20, 0)  (seasons × players)
